In [0]:
import pandas as pd                                                                 # Import pandas for data cleaning
import numpy as np                                                                  # Import Numpy for Maths functions
import matplotlib.pyplot as plt                                                     # Import Matplot for Viz functions
import seaborn as sns                                                               # Import Seaborn for visualization
import plotly.express as px                                                         # Import plotly library for visualization
import plotly.graph_objects as go

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from pyspark.sql import functions as F
from pyspark.sql.functions import col                                               # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev,count,sum as _sum          # MathsFunctions
from pyspark.sql.functions import to_date,year,month,datediff                       # DateFunctions
from pyspark.sql.functions import abs                                               # OtherFunctions

from sklearn.linear_model import LogisticRegression                                 # Classification Analysis Functions
from pyspark.ml.feature import StringIndexer, VectorAssembler                       # Classification Analysis Functions
from sklearn.metrics import mean_squared_error, r2_score                            # Classification Analysis Functions
from sklearn.linear_model import LinearRegression                                   # Classification Analysis Functions
from sklearn.impute import SimpleImputer                                            # Classification Analysis Functions
from sklearn.preprocessing import LabelEncoder                                      # Classification Analysis Functions
from sklearn.preprocessing import OneHotEncoder                                     # Classification Analysis Functions
from sklearn.preprocessing import StandardScaler                                    # Classification Analysis Functions
from sklearn.model_selection import train_test_split                                # Classification Analysis Functions
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report # Classification Analysis Functions
from sklearn.metrics import roc_curve, auc                                          # Classification Analysis Functions
from sklearn.metrics import f1_score                                                # Classification Analysis Functions
from sklearn.tree import DecisionTreeRegressor, plot_tree                           # Classification Analysis Functions

In [0]:
# Load dataset as Spark DataFrame

df = spark.table("lifeexpectancy.silver.life_expectancy")
df = df.toPandas()
df_raw = df
# display(df)

In [0]:
# --- Define Features & Target ---
features = [
    "Adult_Mortality","Alcohol","Hepatitis_B","Measles","BMI",
    "Polio","Total_expenditure","Diphtheria","HIV_AIDS","GDP",
    "Population","Income_composition_of_resources","Schooling"
]
X = df[features]
y = df["Life_expectancy"]

In [0]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 7: Fit Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

# Step 8: Predictions
y_pred = model.predict(X_test)

# Step 9: Evaluation
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)
print("R² Score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

#PRODUCTION
y_full_pred = model.predict(X)

# Save predictions to df
df["LinearPredictions"] = y_full_pred

# Step 9: Evaluation
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)
print("R² Score:", r2_score(y,y_full_pred))
print("RMSE:", np.sqrt(mean_squared_error(y,y_full_pred)))

In [0]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Decision Tree Regressor
model = DecisionTreeRegressor(max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluation
print("Train R²:", r2_score(y_train, y_pred_train))
print("Test R²:", r2_score(y_test, y_pred_test))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

#PRODUCTION-------------------------------------------------------------------------
y_full_pred = model.predict(X)

# Save predictions to df
df["DecisionTreePredictions"] = y_full_pred

# Evaluation
print("Train R²:", r2_score(y_train, y_pred_train))
print("Test R²:", r2_score(y_test, y_pred_test))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))


In [0]:
# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Train Model ---
model = RandomForestRegressor(
    # n_estimators=200,      # number of trees
    # max_depth=10,          # limit depth to avoid overfitting
    # random_state=42,
    # n_jobs=-1              # parallel training
)
model.fit(X_train, y_train)

# --- Predictions ---
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# --- Evaluation ---
print("Train R²:", r2_score(y_train, y_pred_train))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
print("\nTest R²:", r2_score(y_test, y_pred_test))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

#PRODUCTION-----------------------------------------------------------------------

model = RandomForestRegressor()
model.fit(X, y)

y_full_pred = model.predict(X)

# Save predictions to df
df["RandomForestPredictions"] = y_full_pred

# Evaluation
print("Train R²:", r2_score(y_train, y_pred_train))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
print("\nTest R²:", r2_score(y_test, y_pred_test))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
#

In [0]:
df = spark.createDataFrame(df)
df.write.mode("overwrite").saveAsTable("lifeexpectancy.silver.life_expectancy_Predictions")